# PF200 ORB ML Research V2

Objetivo: entrenar clasificadores futures-native por slot y lado para ORB1/ORB2 sobre MNQ/MES.

Principios:
- no reemplazar el gobernador determinístico
- entrenar offline en QC Research
- persistir bundles por slot/lado en ObjectStore
- integración futura solo si mejora IS/OOS/STRESS sin breaches


In [ ]:
import warnings, json, math, pickle, zlib, base64
warnings.filterwarnings("ignore")
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from AlgorithmImports import *
from QuantConnect.Research import QuantBook

try:
    from lightgbm import LGBMClassifier
    HAVE_LGBM = True
except Exception:
    HAVE_LGBM = False

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
qb = QuantBook()
print("LGBM available:", HAVE_LGBM)


In [ ]:
CONFIG = {
    "start": datetime(2020, 1, 1),
    "end": datetime(2026, 3, 31),
    "or_minutes": 15,
    "slots": [("ORB1", 10, 15), ("ORB2", 11, 0)],
    "fwd_minutes": 120,
    "stop_atr_mult": 0.75,
    "target_atr_mult": 1.55,
    "symbols": ["MNQ", "MES"],
    "objectstore_prefix": "pf200_orb_ml_v2/",
    "threshold_grid": [0.50, 0.55, 0.60, 0.65],
}
CONFIG


In [ ]:
future_map = {
    "MNQ": Futures.Indices.MicroNASDAQ100EMini,
    "MES": Futures.Indices.MicroSP500EMini,
}
contracts = {}
for name, future_type in future_map.items():
    fut = qb.AddFuture(future_type, Resolution.Minute)
    fut.SetFilter(0, 182)
    contracts[name] = fut.Symbol
contracts


## Dataset builder
Construye ejemplos por día, símbolo y slot con features alineadas con el motor actual.


In [ ]:
def _flatten_history(df):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.reset_index()
    out.columns = [str(c).lower() for c in out.columns]
    return out

def fetch_histories(symbol, start, end):
    min_df = _flatten_history(qb.History(symbol, start, end, Resolution.Minute))
    day_df = _flatten_history(qb.History(symbol, start - timedelta(days=120), end, Resolution.Daily))
    return min_df, day_df

def atr14(day_df):
    d = day_df.copy()
    d["prev_close"] = d["close"].shift(1)
    tr = pd.concat([
        (d["high"] - d["low"]).abs(),
        (d["high"] - d["prev_close"]).abs(),
        (d["low"] - d["prev_close"]).abs(),
    ], axis=1).max(axis=1)
    return tr.rolling(14).mean()

def realized_vol20(day_df):
    ret = day_df["close"].pct_change()
    return ret.rolling(20).std() * np.sqrt(252)

def build_dataset_for_symbol(name, symbol, cfg):
    min_df, day_df = fetch_histories(symbol, cfg["start"], cfg["end"])
    if min_df.empty or day_df.empty:
        return pd.DataFrame()
    min_df["time"] = pd.to_datetime(min_df["time"])
    min_df["date"] = min_df["time"].dt.date
    day_df["time"] = pd.to_datetime(day_df["time"])
    day_df["date"] = day_df["time"].dt.date
    day_df["atr14"] = atr14(day_df)
    day_df["trend55"] = day_df["close"].ewm(span=55, adjust=False).mean()
    day_df["rv20"] = realized_vol20(day_df)
    day_df["prev_range_pct"] = ((day_df["high"] - day_df["low"]) / day_df["close"].shift(1)).replace([np.inf, -np.inf], np.nan)
    daily = day_df[["date","open","high","low","close","atr14","trend55","rv20","prev_range_pct"]].copy()
    rows = []
    for d, intraday in min_df.groupby("date"):
        if len(intraday) < 100:
            continue
        ref = daily[daily["date"] < d].tail(1)
        cur = daily[daily["date"] == d].tail(1)
        if ref.empty or cur.empty:
            continue
        prev_close = float(ref.iloc[0]["close"])
        atr = float(cur.iloc[0]["atr14"]) if pd.notna(cur.iloc[0]["atr14"]) else np.nan
        trend = float(cur.iloc[0]["trend55"]) if pd.notna(cur.iloc[0]["trend55"]) else np.nan
        rv20 = float(cur.iloc[0]["rv20"]) if pd.notna(cur.iloc[0]["rv20"]) else np.nan
        prev_range_pct = float(cur.iloc[0]["prev_range_pct"]) if pd.notna(cur.iloc[0]["prev_range_pct"]) else np.nan
        if not np.isfinite(prev_close) or not np.isfinite(atr) or atr <= 0:
            continue
        intraday = intraday.sort_values("time").copy()
        session_open = float(intraday.iloc[0]["open"])
        gap_pct = (session_open - prev_close) / prev_close if prev_close > 0 else 0.0
        open_ts = pd.Timestamp(datetime.combine(pd.Timestamp(d).date(), datetime.min.time()) + timedelta(hours=9, minutes=30))
        or_end = open_ts + timedelta(minutes=cfg["or_minutes"])
        or_bars = intraday[intraday["time"] <= or_end]
        if len(or_bars) < 3:
            continue
        or_high = float(or_bars["high"].max())
        or_low = float(or_bars["low"].min())
        or_width = or_high - or_low
        or_width_atr = or_width / atr
        for slot_name, hh, mm in cfg["slots"]:
            slot_ts = pd.Timestamp(datetime.combine(pd.Timestamp(d).date(), datetime.min.time()) + timedelta(hours=hh, minutes=mm))
            now_bars = intraday[intraday["time"] <= slot_ts]
            if now_bars.empty:
                continue
            px = float(now_bars.iloc[-1]["close"])
            day_high = max(float(now_bars["high"].max()), px)
            day_low = min(float(now_bars["low"].min()), px)
            day_range_atr = (day_high - day_low) / atr
            intraday_mom = (px - session_open) / session_open if session_open > 0 else 0.0
            long_break = float(px > or_high)
            short_break = float(px < or_low)
            uptrend = float(px > trend) if np.isfinite(trend) else 0.0
            downtrend = float(px < trend) if np.isfinite(trend) else 0.0
            future_bars = intraday[(intraday["time"] > slot_ts) & (intraday["time"] <= slot_ts + timedelta(minutes=cfg["fwd_minutes"]))]
            if future_bars.empty:
                continue
            stop_dist = atr * cfg["stop_atr_mult"]
            target_dist = atr * cfg["target_atr_mult"]
            max_up = float(future_bars["high"].max()) - px
            max_dn = px - float(future_bars["low"].min())
            y_long = int(max_up >= target_dist and max_dn <= stop_dist) if stop_dist > 0 else 0
            y_short = int(max_dn >= target_dist and max_up <= stop_dist) if stop_dist > 0 else 0
            rows.append({
                "symbol": name,
                "date": pd.Timestamp(d),
                "slot": slot_name,
                "px": px,
                "prev_close": prev_close,
                "session_open": session_open,
                "gap_pct": gap_pct,
                "or_width_atr": or_width_atr,
                "day_range_atr": day_range_atr,
                "intraday_mom": intraday_mom,
                "long_break": long_break,
                "short_break": short_break,
                "uptrend": uptrend,
                "downtrend": downtrend,
                "rv20": rv20,
                "prev_range_pct": prev_range_pct,
                "target_dist": target_dist,
                "stop_dist": stop_dist,
                "y_long": y_long,
                "y_short": y_short,
            })
    return pd.DataFrame(rows)


In [ ]:
frames = []
for name, symbol in contracts.items():
    df = build_dataset_for_symbol(name, symbol, CONFIG)
    print(name, df.shape)
    frames.append(df)
dataset = pd.concat([f for f in frames if not f.empty], ignore_index=True) if frames else pd.DataFrame()
dataset.head(), dataset.shape


In [ ]:
FEATURES = [
    "gap_pct", "or_width_atr", "day_range_atr", "intraday_mom",
    "long_break", "short_break", "uptrend", "downtrend", "rv20", "prev_range_pct"
]
dataset["date"] = pd.to_datetime(dataset["date"])
SPLITS = {
    "IS": (pd.Timestamp("2022-01-01"), pd.Timestamp("2024-12-31")),
    "OOS": (pd.Timestamp("2025-01-01"), pd.Timestamp("2026-03-31")),
    "STRESS": (pd.Timestamp("2020-01-01"), pd.Timestamp("2020-12-31")),
}

def pick_model():
    if HAVE_LGBM:
        model = LGBMClassifier(n_estimators=250, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42)
    else:
        model = RandomForestClassifier(n_estimators=400, max_depth=5, min_samples_leaf=20, random_state=42, n_jobs=-1)
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", model)])

def eval_threshold(y_true, probs, thresholds):
    best = None
    for thr in thresholds:
        pred = (probs >= thr).astype(int)
        precision = precision_score(y_true, pred, zero_division=0)
        recall = recall_score(y_true, pred, zero_division=0)
        signal_rate = float(pred.mean())
        score = precision * max(signal_rate, 1e-6)
        row = {"threshold": thr, "precision": float(precision), "recall": float(recall), "signal_rate": signal_rate, "score": score}
        if best is None or row["score"] > best["score"]:
            best = row
    return best


In [ ]:
results = []
bundles = {}
for slot in sorted(dataset["slot"].dropna().unique()):
    for side, target_col in [("LONG", "y_long"), ("SHORT", "y_short")]:
        sub = dataset[dataset["slot"] == slot].copy()
        train = sub[(sub["date"] >= SPLITS["IS"][0]) & (sub["date"] <= SPLITS["IS"][1])].copy()
        if train.empty or train[target_col].nunique() < 2:
            print("skip", slot, side, "insufficient train data")
            continue
        pipe = pick_model()
        pipe.fit(train[FEATURES], train[target_col])
        train_probs = pipe.predict_proba(train[FEATURES])[:,1]
        picked = eval_threshold(train[target_col].values, train_probs, CONFIG["threshold_grid"])
        metrics = {}
        for split_name, (a,b) in SPLITS.items():
            frame = sub[(sub["date"] >= a) & (sub["date"] <= b)].copy()
            if frame.empty or frame[target_col].nunique() < 2:
                metrics[split_name] = {"n": int(len(frame)), "auc": None, "precision": None, "signal_rate": None}
                continue
            probs = pipe.predict_proba(frame[FEATURES])[:,1]
            pred = (probs >= picked["threshold"]).astype(int)
            auc = roc_auc_score(frame[target_col], probs) if frame[target_col].nunique() > 1 else np.nan
            precision = precision_score(frame[target_col], pred, zero_division=0)
            signal_rate = float(pred.mean())
            metrics[split_name] = {"n": int(len(frame)), "auc": float(auc) if np.isfinite(auc) else None, "precision": float(precision), "signal_rate": signal_rate}
        bundle_key = f"{slot}_{side}".lower()
        bundle = {
            "version": "pf200_orb_ml_v2",
            "slot": slot,
            "side": side,
            "features": FEATURES,
            "target_col": target_col,
            "threshold": picked["threshold"],
            "metrics": metrics,
            "model_bytes_b64": base64.b64encode(zlib.compress(pickle.dumps(pipe, protocol=pickle.HIGHEST_PROTOCOL), 9)).decode("ascii"),
        }
        bundles[bundle_key] = bundle
        results.append({"slot": slot, "side": side, "picked": picked, "metrics": metrics})

results_df = pd.DataFrame([{**{"slot": r["slot"], "side": r["side"]}, **{f"{k}_{m}": v2 for k,v in r["metrics"].items() for m,v2 in v.items()}} for r in results]) if results else pd.DataFrame()
results_df


In [ ]:
manifest = {
    "version": "pf200_orb_ml_v2",
    "created_utc": datetime.utcnow().isoformat(),
    "features": FEATURES,
    "config": CONFIG,
    "bundle_keys": sorted(list(bundles.keys())),
}
base_prefix = CONFIG["objectstore_prefix"]
for key, bundle in bundles.items():
    qb.ObjectStore.Save(base_prefix + key + ".json", json.dumps(bundle))
qb.ObjectStore.Save(base_prefix + "manifest.json", json.dumps(manifest, indent=2))
print("saved bundle keys:", sorted(bundles.keys()))
manifest
